# 00-7. 文字の飾りと書体 — 動かして確かめる

📖 解説: [`../07_decorations_and_fonts.md`](../07_decorations_and_fonts.md)

上から順に **Shift+Enter** で実行していってください。

## このノートで触るもの
1. サイコロで $\mu$ (真の値) と $\bar{x}$ (標本平均) の違いを体感
2. 【対話】データを増やすと $\bar{x}$ が $\mu$ に近づく (大数の法則)
3. 【対話】推定値 $\hat{\theta}$ はどれくらいブレるのか
4. ⚠️ 同じハットが 3 つの意味 — 単位ベクトル / 推定量 / 予測値
5. 太字と shape の対応
6. $x^{(i)}$ は累乗ではない
7. コードでの命名の慣習

> 🧭 **クイックナビ**: 📚 [ROOT (全体 TOP)](../../README.md) ・ 🏠 [章 TOP](../README.md) ・ 📖 [解説 md (07_decorations_and_fonts.md)](../07_decorations_and_fonts.md)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore", message=".*distutils Version classes.*", category=DeprecationWarning)
import japanize_matplotlib  # noqa: F401  # 日本語フォント (豆腐化対策)
from ipywidgets import interact, IntSlider

# 乱数の種を固定すると毎回同じ結果になる (再現性のため)
rng = np.random.default_rng(42)

## 1. $\mu$ と $\bar{x}$ — 真の値と、手元のデータ

公平なサイコロの**真の平均**は計算で分かります:

$$
\mu = \frac{1+2+3+4+5+6}{6} = 3.5
$$

では実際に 10 回振ったらどうなるか。やってみましょう。

In [ ]:
# 真の値 (今回は計算で分かっている)
mu_true: float = 3.5  # μ  真の平均 (単位: 出目)

# 実際に 10 回振る
rolls: np.ndarray = rng.integers(1, 7, size=10)  # shape: (10,)

# データから推定した値
x_bar: float = float(rolls.mean())  # x̄  標本平均
mu_hat: float = x_bar               # μ̂  μ の推定値

print(f'出た目        : {rolls}')
print(f'μ  (真の値)   : {mu_true}')
print(f'x̄ (標本平均) : {x_bar}')
print(f'推定誤差       : {abs(mu_hat - mu_true):.2f}')

### もう一度振ってみる

**同じコードをもう一度実行**してみてください（下のセルを何度か Shift+Enter）。

$\bar{x}$ の値が**毎回変わる**はずです。これが「$\bar{x}$ は推定値であってブレるもの」ということ。
一方 $\mu = 3.5$ は絶対に変わりません。それがサイコロの本当の性質だからです。

In [ ]:
# 何度か実行してみてください (実行のたびに値が変わります)
for trial in range(5):
    sample = rng.integers(1, 7, size=10)   # shape: (10,)
    print(f'{trial + 1} 回目の実験: x̄ = {sample.mean():.2f}   (μ = 3.5 との差: {abs(sample.mean() - 3.5):+.2f})')

## 2. 【対話】大数の法則 — データを増やすと $\bar{x} \to \mu$

スライダーでサイコロを振る回数 $n$ を変えてみてください。

**注目点**: $n$ が小さいと $\bar{x}$ は 3.5 から大きく外れますが、
$n$ を増やすほど赤い線（真の値 3.5）に吸い寄せられていきます。

In [ ]:
def plot_convergence(n: int = 10) -> None:
    """サイコロを n 回振り、累積平均が μ=3.5 に近づく様子を描く.

    Args:
        n: 振る回数
    """
    local_rng = np.random.default_rng(0)          # 途中経過を見たいので種を固定
    rolls = local_rng.integers(1, 7, size=n)      # shape: (n,)
    # 1 回目まで、2 回目まで、… の平均 (累積平均)
    running_mean = np.cumsum(rolls) / np.arange(1, n + 1)   # shape: (n,)

    plt.figure(figsize=(9, 4.5))
    plt.plot(np.arange(1, n + 1), running_mean, lw=1.5, label=r'標本平均 $\bar{x}$ (累積)')
    plt.axhline(3.5, c='r', ls='--', lw=2, label=r'真の値 $\mu = 3.5$')
    plt.xlabel('振った回数 n')
    plt.ylabel('平均')
    plt.ylim(1, 6)
    plt.title(f'n = {n} 回振ったときの標本平均: {running_mean[-1]:.4f}')
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()

    print(f'n = {n:>6}   x̄ = {running_mean[-1]:.4f}   |x̄ - μ| = {abs(running_mean[-1] - 3.5):.4f}')


interact(plot_convergence, n=IntSlider(min=10, max=5000, step=10, value=10))

## 3. 【対話】推定値 $\hat{\theta}$ はどれくらいブレるのか

同じ実験を何度も繰り返すと、$\hat{\theta}$ は毎回違う値になります。
その**ばらつき方**を見てみましょう。

**注目点**: サンプル数 $n$ を増やすと、ヒストグラムの幅が**狭く**なります。
「たくさんデータを取れば推定が安定する」ということが目で見えます。

In [ ]:
def plot_estimator_spread(n: int = 10) -> None:
    """サイコロ n 回の実験を 2000 回繰り返し、x̄ の分布を描く.

    Args:
        n: 1 回の実験で振る回数
    """
    n_experiments: int = 2000                          # 実験の繰り返し回数
    local_rng = np.random.default_rng(1)
    samples = local_rng.integers(1, 7, size=(n_experiments, n))  # shape: (2000, n)
    x_bars = samples.mean(axis=1)                                # shape: (2000,)

    plt.figure(figsize=(9, 4.5))
    plt.hist(x_bars, bins=40, alpha=0.75, edgecolor='white')
    plt.axvline(3.5, c='r', ls='--', lw=2, label=r'真の値 $\mu = 3.5$')
    plt.xlim(1, 6)
    plt.xlabel(r'標本平均 $\bar{x}$')
    plt.ylabel('回数')
    plt.title(f'n = {n} の実験を 2000 回: 推定値のばらつき (標準偏差 {x_bars.std():.3f})')
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()

    print(f'n = {n:>4}   x̄ の平均 = {x_bars.mean():.4f} (μ=3.5 に近い)   x̄ のばらつき = {x_bars.std():.4f}')


interact(plot_estimator_spread, n=IntSlider(min=1, max=200, step=1, value=10))

> 💡 **2 つのことが同時に見えています。**
> 1. ヒストグラムの**中心**は $n$ によらず 3.5 → 推定量が偏っていない（**不偏性**）
> 2. ヒストグラムの**幅**は $n$ を増やすと狭くなる → データが多いほど信頼できる
>
> この 2 点が、[`03_probability_statistics/`](../../03_probability_statistics/README.md) で学ぶ推定論の出発点です。

## 4. ⚠️ 同じハットが 3 つの意味

このリポジトリの中だけでも、ハットは 3 通りに使われています。
**実際に計算して**、まったく別物であることを確認しましょう。

In [ ]:
# --- 意味 1: 単位ベクトル (線形代数) ---
# v̂ = v / ||v||   長さを 1 に揃えたベクトル
v: np.ndarray = np.array([3.0, 4.0])           # shape: (2,)
v_hat: np.ndarray = v / np.linalg.norm(v)      # v̂

print('【意味1】単位ベクトル')
print(f'  v      = {v}          長さ = {np.linalg.norm(v):.1f}')
print(f'  v̂     = {v_hat}   長さ = {np.linalg.norm(v_hat):.1f}  ← 長さが 1 になった')

In [ ]:
# --- 意味 2: 推定量 (統計) ---
# θ̂ = データから推定したパラメータ
theta_true: float = 0.3                                  # θ  真の「表が出る確率」
coin: np.ndarray = rng.random(50) < theta_true           # shape: (50,) 表なら True
theta_hat: float = float(coin.mean())                    # θ̂ 推定値

print('【意味2】推定量')
print(f'  θ  (真の確率) = {theta_true}')
print(f'  θ̂ (推定値)   = {theta_hat:.3f}  ← 50 回投げて数えた結果')
print('     (実行するたびに変わります。たまたま真値と一致することもある)')

In [ ]:
# --- 意味 3: 予測値 (機械学習) ---
# ŷ = モデルの出力、y = 正解
x_data: np.ndarray = np.array([1.0, 2.0, 3.0, 4.0])      # shape: (4,) 入力
y_true: np.ndarray = np.array([2.1, 3.9, 6.2, 7.8])      # shape: (4,) 正解 y
w, b = 2.0, 0.0                                          # モデルのパラメータ
y_hat: np.ndarray = w * x_data + b                       # ŷ 予測値

print('【意味3】予測値')
print(f'  y  (正解)   = {y_true}')
print(f'  ŷ (予測)   = {y_hat}')
print(f'  誤差        = {np.abs(y_true - y_hat)}')
print()
print('→ 3 つとも「ハット」だが、長さ 1 / 推定 / 予測 とまったく別の意味。')
print('  見分け方: 太字のベクトルなら単位ベクトル、正解 y と対なら予測値。')

## 5. 太字と shape の対応

$$
\mathbf{y} = \mathbf{A}\mathbf{x}
$$

太字を見るだけで「行列 × ベクトル = ベクトル」と読めます。実際に確認しましょう。

In [ ]:
x_scalar: float = 3.0                                  # x   スカラー
x_vec: np.ndarray = np.array([1.0, 0.0, -1.0])         # 𝐱   ベクトル shape: (3,)
A_mat: np.ndarray = np.array([[1.0, 2.0, 3.0],
                              [4.0, 5.0, 6.0]])        # 𝐀   行列    shape: (2, 3)

y_vec: np.ndarray = A_mat @ x_vec                       # 𝐲 = 𝐀𝐱   shape: (2,)

print(f'x      (スカラー) : {x_scalar}          type  = {type(x_scalar).__name__}')
print(f'𝐱     (ベクトル) : {x_vec}   shape = {x_vec.shape}')
print(f'𝐀     (行列)     : shape = {A_mat.shape}')
print(f'𝐲 = 𝐀𝐱          : {y_vec}       shape = {y_vec.shape}')
print()
print(f'  (2, 3) @ (3,) -> (2,)   ← 数式を見た瞬間にこれが読めるのが目標')

### $\mathbf{x} \in \mathbb{R}^n$ は「shape 宣言」

論文の $\mathbf{x} \in \mathbb{R}^{n}$, $W \in \mathbb{R}^{m \times n}$ は、
コードでいえば `x.shape == (n,)`, `W.shape == (m, n)` の宣言です。

**shape が合わないとどうなるか**、わざとエラーを出して確認しましょう。

In [ ]:
# わざと shape を間違えてみる
wrong: np.ndarray = np.array([1.0, 2.0])   # shape: (2,)  ← A は (2,3) なので合わない

try:
    A_mat @ wrong
except ValueError as e:
    print('❌ ValueError が出ました:')
    print(f'   {e}')
    print()
    print('→ 数式を読む段階で shape を把握しておくと、この手のバグを防げます。')
    print(f'   A は {A_mat.shape}、掛けられるのは shape ({A_mat.shape[1]},) のベクトルだけ。')

## 6. $x^{(i)}$ は累乗ではない

機械学習の論文にはこう書かれます:

$$
\mathcal{L} = \frac{1}{N}\sum_{i=1}^{N}\left(y^{(i)} - \hat{y}^{(i)}\right)^2
$$

$y^{(i)}$ は「$y$ の $i$ 乗」ではなく **$i$ 番目のデータ**。上付きカッコがその印です。

In [ ]:
# X.shape == (N, d)   N 個のサンプル、各サンプルは d 次元
X: np.ndarray = np.array([[1.0, 2.0],
                          [3.0, 4.0],
                          [5.0, 6.0]])   # shape: (3, 2)  N=3, d=2

i, j = 1, 0
print(f'X         = shape {X.shape}   (N={X.shape[0]} サンプル, d={X.shape[1]} 次元)')
print(f'x^(i)     = X[{i}]     = {X[i]}     ← {i} 番目の「サンプル」 (累乗ではない)')
print(f'x^(i)_j   = X[{i}, {j}]  = {X[i, j]}          ← {i} 番目サンプルの {j} 番目の特徴量')
print()
print(f'ちなみに 2 の 3 乗は {2**3} — こちらが本物の累乗')

## 7. コードでの命名の慣習

数式の飾りは、Python では**変数名**で表現します。

| 数式 | Python |
|---|---|
| $\theta$ | `theta` / `theta_true` |
| $\hat{\theta}$ | `theta_hat` |
| $\bar{x}$ | `x_bar` / `x_mean` |
| $\theta^*$ | `theta_star` / `theta_opt` |
| $\tilde{x}$ | `x_tilde` |
| $\hat{y}$ | `y_pred` (ML では慣習的にこちら) |
| $y$ | `y_true` |

In [ ]:
# 数式とコードの対応をコメントで残しておくと後で読みやすい
theta_true: float = 0.30      # θ    真のパラメータ
theta_hat: float = 0.28       # θ̂   推定値
x_bar: float = 3.42           # x̄   標本平均
theta_star: float = 0.35      # θ*   最適値

print(f'θ  = {theta_true}')
print(f'θ̂ = {theta_hat}   誤差 {abs(theta_hat - theta_true):.3f}')
print(f'x̄ = {x_bar}')
print(f'θ* = {theta_star}')
print()
print('→ コメントに元の記号を書いておくと、論文とコードを行き来しやすくなります。')

## まとめ

- $\mu$ は**真の値**（変わらない）、$\bar{x}$ は**手元のデータの平均**（実行のたびにブレる）
- **ハットは「これは推定値であって本物ではない」という宣言**
- データを増やすと $\bar{x} \to \mu$（大数の法則）、推定のばらつきも小さくなる
- **同じハットでも分野で意味が違う** — 単位ベクトル / 推定量 / 予測値
- **太字かどうかで shape が決まる** — $x$ はスカラー、$\mathbf{x}$ は `(n,)`、$\mathbf{A}$ は `(m,n)`
- $x^{(i)}$ の上付きカッコは**サンプル番号**（累乗ではない）
- コードでは `theta_hat` / `x_bar` のように変数名で区別する

## 記号読解章、卒業 🎉

これで [`00_notation/`](../README.md) は修了です。論文の数式を見ても、記号に怯まなくなったはずです。

→ 次の章: [`../../01_linear_algebra/README.md`](../../01_linear_algebra/README.md) — 太字の $\mathbf{x}$ と $\mathbf{A}$ が主役になる世界へ

---

## 📍 ナビゲーション

| ← 前 | 🏠 章 TOP | 📚 全体 TOP | 次の章 → |
|---|---|---|---|
| [`06_greek_letters.ipynb`](06_greek_letters.ipynb) | [章 TOP](../README.md) | [📚 ROOT README](../../README.md) | [`../../01_linear_algebra/README.md`](../../01_linear_algebra/README.md) |